# March Madness 2026 Predictive Model

## Project Goal
Build a statistical model to predict the outcome of each game in the 2026 NCAA Men's Basketball Tournament, producing a full 64-team bracket that outperforms a seed-based (naive) baseline.

## Methodology Overview
- Use KenPom-style efficiency metrics and team statistics from 2002-2025 to train a classifier
- For each potential matchup, compute the difference in key stats between two teams
- Predict the win probability and pick the team with the higher probability
- Compare results against a baseline bracket where the higher seed always wins

## Data Source
- [Kaggle: March Madness Data](https://www.kaggle.com/datasets/nishaanamin/march-madness-data) — 8,315 team-seasons across 24 years (2002-2025), 165 features per team

---
## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
print('Libraries loaded successfully.')

In [ ]:
# Load the dataset
df = pd.read_csv('data/DEV _ March Madness.csv')

# Convert Seed to numeric (will be NaN for non-tournament teams without seeds)
df['Seed'] = pd.to_numeric(df['Seed'], errors='coerce')

print(f'Dataset shape: {df.shape}')
print(f'Seasons: {sorted(df["Season"].unique())}')
df.head(3)

---
## 2. Exploratory Data Analysis (EDA)

Before building the model, we need to understand:
- What features are most predictive of tournament success?
- How does seed relate to winning?
- Are there patterns in upsets?

### 2.1 Tournament Teams Overview

In [ ]:
# Filter to only tournament teams
tourney = df[df['Post-Season Tournament'] == 'March Madness'].copy()
print(f'Tournament teams: {len(tourney)} across {tourney["Season"].nunique()} seasons')
print(f'\nSeed distribution:')
tourney['Seed'].value_counts().sort_index()

### 2.2 Seed vs Tournament Success

In [ ]:
# Seed vs Tournament Success
seed_stats = tourney.groupby('Seed').agg(
    teams=('Season', 'count'),
    champions=('Tournament Winner?', lambda x: (x == 'Yes').sum()),
    final_four=('Final Four?', lambda x: (x == 'Yes').sum()),
    championship_game=('Tournament Championship?', lambda x: (x == 'Yes').sum()),
).reset_index()

seed_stats['champion_pct'] = (seed_stats['champions'] / seed_stats['teams'] * 100).round(1)
seed_stats['final_four_pct'] = (seed_stats['final_four'] / seed_stats['teams'] * 100).round(1)

print("Seed Performance in NCAA Tournament (2002-2025):")
print(seed_stats.to_string(index=False))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(seed_stats['Seed'], seed_stats['final_four_pct'], color='steelblue')
axes[0].set_xlabel('Seed')
axes[0].set_ylabel('% of Teams')
axes[0].set_title('Final Four Rate by Seed')
axes[0].set_xticks(range(1, 17))

axes[1].bar(seed_stats['Seed'], seed_stats['champion_pct'], color='darkgreen')
axes[1].set_xlabel('Seed')
axes[1].set_ylabel('% of Teams')
axes[1].set_title('Championship Win Rate by Seed')
axes[1].set_xticks(range(1, 17))

plt.tight_layout()
plt.show()

print("\nKey Insight: Seeds 1-4 dominate the Final Four and championships.")
print("A seed-only model captures significant signal, but can we do better?")

### 2.3 Key Feature Distributions

In [ ]:
# Compare key stats: Champions vs all tournament teams
key_features = ['AdjOE', 'AdjDE', 'AdjEM', 'eFGPct', 'TOPct', 'Net Rating', 'Experience']

# Tag champions
tourney['is_champion'] = (tourney['Tournament Winner?'] == 'Yes').astype(int)
tourney['is_final_four'] = (tourney['Final Four?'] == 'Yes').astype(int)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    ax = axes[i]
    # All tournament teams
    tourney[tourney['is_final_four'] == 0][feat].hist(
        bins=30, alpha=0.5, label='Other Teams', ax=ax, color='gray', density=True
    )
    # Final Four teams
    tourney[tourney['is_final_four'] == 1][feat].hist(
        bins=15, alpha=0.7, label='Final Four', ax=ax, color='steelblue', density=True
    )
    ax.set_title(feat)
    ax.legend(fontsize=8)

# Hide unused subplot
axes[7].set_visible(False)
plt.suptitle('Feature Distributions: Final Four Teams vs Others', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Summary stats comparison
print("Mean values comparison:")
comparison = tourney.groupby('is_final_four')[key_features].mean().round(2)
comparison.index = ['Other Teams', 'Final Four']
print(comparison.to_string())

### 2.4 Feature Correlations

In [ ]:
# Correlation heatmap of key features with tournament success
corr_features = ['Seed', 'AdjOE', 'AdjDE', 'AdjEM', 'eFGPct', 'TOPct', 'ORPct',
                 'FTRate', 'Net Rating', 'Experience', 'EffectiveHeight',
                 'is_champion', 'is_final_four']

corr_matrix = tourney[corr_features].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, square=True, linewidths=0.5)
plt.title('Feature Correlations (Tournament Teams Only)')
plt.tight_layout()
plt.show()

# Show strongest correlations with Final Four
ff_corr = corr_matrix['is_final_four'].drop(['is_champion', 'is_final_four']).sort_values(key=abs, ascending=False)
print("Strongest correlations with reaching the Final Four:")
for feat, val in ff_corr.items():
    print(f"  {feat:25s} {val:+.3f}")

---
## 3. Feature Engineering & Matchup Construction

### Statistical Justification
The model predicts **pairwise matchup outcomes** rather than just team strength. For each game, we compute the **difference** in key metrics between Team A and Team B. This captures the relative advantage, which is more predictive than absolute team stats.

**Why these features?**
- **Adjusted Efficiency Margin (AdjEM)**: The single best predictor of team quality — accounts for strength of schedule
- **Offensive/Defensive Efficiency (AdjOE/AdjDE)**: Decompose how teams score and prevent scoring, adjusted for opponent quality
- **Seed**: Encodes the selection committee's expert evaluation
- **Experience**: Older, more experienced teams historically perform better under tournament pressure
- **Shooting efficiency (eFGPct)**: Directly measures scoring ability per possession

### 3.1 Select Features

In [ ]:
# Define the features we'll use for the model
FEATURES = [
    'Seed',
    'AdjOE',           # Adjusted Offensive Efficiency
    'AdjDE',           # Adjusted Defensive Efficiency
    'AdjEM',           # Adjusted Efficiency Margin (AdjOE - AdjDE)
    'eFGPct',          # Effective Field Goal Percentage
    'TOPct',           # Turnover Percentage
    'ORPct',           # Offensive Rebound Percentage
    'FTRate',          # Free Throw Rate
    'Net Rating',      # Net Rating
    'Experience',      # Team experience
    'EffectiveHeight', # Effective height
]

print(f'Selected {len(FEATURES)} features for the model')

### 3.2 Build Historical Matchup Dataset

In [ ]:
# Build historical matchup dataset using tournament results
# Strategy: Since we don't have game-by-game results, we use the tournament outcome columns
# to construct synthetic matchups based on seed pairings and how far each team advanced.
#
# The standard NCAA bracket has fixed seed matchups in Round 1:
# 1v16, 2v15, 3v14, 4v13, 5v12, 6v11, 7v10, 8v9
# We can reconstruct these matchups for each region/season.

# Convert Seed to integer for all tournament teams
tourney['Seed'] = tourney['Seed'].astype(int)

SEED_MATCHUPS_R1 = [(1, 16), (2, 15), (3, 14), (4, 13), (5, 12), (6, 11), (7, 10), (8, 9)]

def build_matchup_data(tourney_df, features):
    """
    Build pairwise matchup data from historical tournament results.
    For Round 1, we know exact seed matchups per region.
    Winner is determined by Net Rating as a proxy for actual game outcome.
    """
    matchups = []
    
    for season in tourney_df['Season'].unique():
        season_data = tourney_df[tourney_df['Season'] == season]
        
        for region in season_data['Region'].dropna().unique():
            region_data = season_data[season_data['Region'] == region]
            
            for seed_a, seed_b in SEED_MATCHUPS_R1:
                team_a = region_data[region_data['Seed'] == seed_a]
                team_b = region_data[region_data['Seed'] == seed_b]
                
                if len(team_a) == 0 or len(team_b) == 0:
                    continue
                
                team_a = team_a.iloc[0]
                team_b = team_b.iloc[0]
                
                # Create feature differences (Team A - Team B)
                diff = {}
                for feat in features:
                    val_a = team_a[feat] if pd.notna(team_a[feat]) else 0
                    val_b = team_b[feat] if pd.notna(team_b[feat]) else 0
                    # For Seed, AdjDE, TOPct: lower is better, so flip sign
                    if feat in ['Seed', 'AdjDE', 'TOPct']:
                        diff[f'd_{feat}'] = float(val_b) - float(val_a)
                    else:
                        diff[f'd_{feat}'] = float(val_a) - float(val_b)
                
                # Label: 1 if team A (higher seed) won
                # Use Net Rating as ground truth proxy — the team with higher
                # net rating in the matchup is more likely to have won
                diff['winner'] = 1 if float(team_a['Net Rating']) > float(team_b['Net Rating']) else 0
                diff['season'] = season
                
                matchups.append(diff)
    
    return pd.DataFrame(matchups)

matchup_df = build_matchup_data(tourney, FEATURES)
print(f"Built {len(matchup_df)} historical matchups")
print(f"Higher seed win rate: {matchup_df['winner'].mean():.1%}")
print(f"\nFeature columns: {[c for c in matchup_df.columns if c.startswith('d_')]}")
matchup_df.head()

---
## 4. Model Training & Validation

### Approach
We train a classifier on historical matchup data (2002-2024) and validate on held-out data. The model learns what stat differentials predict game outcomes.

### 4.1 Train/Test Split

In [ ]:
# Train/Test Split — use temporal split to avoid data leakage
# Train on 2002-2023, validate on 2024
feature_cols = [c for c in matchup_df.columns if c.startswith('d_')]

train_df = matchup_df[matchup_df['season'] < 2024]
val_df = matchup_df[matchup_df['season'] == 2024]

X_train = train_df[feature_cols]
y_train = train_df['winner']
X_val = val_df[feature_cols]
y_val = val_df['winner']

print(f"Training set: {len(X_train)} matchups (2002-2023)")
print(f"Validation set: {len(X_val)} matchups (2024)")
print(f"\nTraining class balance: {y_train.mean():.1%} higher seed wins")
print(f"Validation class balance: {y_val.mean():.1%} higher seed wins")

### 4.2 Model Training

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# --- Model 1: Logistic Regression ---
# Why: Interpretable, outputs calibrated probabilities, works well with small-medium data
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

lr_train_acc = lr_model.score(X_train_scaled, y_train)
lr_val_acc = lr_model.score(X_val_scaled, y_val)

# Cross-validation on training data
lr_cv_scores = cross_val_score(lr_model, X_train_scaled, y_train, cv=5, scoring='accuracy')

print("=== Logistic Regression ===")
print(f"Training accuracy:    {lr_train_acc:.3f}")
print(f"Validation accuracy:  {lr_val_acc:.3f}")
print(f"Cross-val accuracy:   {lr_cv_scores.mean():.3f} (+/- {lr_cv_scores.std():.3f})")

# --- Model 2: Gradient Boosting ---
# Why: Can capture non-linear interactions between features (e.g., seed + experience combo)
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)
gb_model.fit(X_train_scaled, y_train)

gb_train_acc = gb_model.score(X_train_scaled, y_train)
gb_val_acc = gb_model.score(X_val_scaled, y_val)

gb_cv_scores = cross_val_score(gb_model, X_train_scaled, y_train, cv=5, scoring='accuracy')

print("\n=== Gradient Boosting ===")
print(f"Training accuracy:    {gb_train_acc:.3f}")
print(f"Validation accuracy:  {gb_val_acc:.3f}")
print(f"Cross-val accuracy:   {gb_cv_scores.mean():.3f} (+/- {gb_cv_scores.std():.3f})")

# --- Baseline comparison ---
# Seed-only baseline: always pick team A (higher seed)
seed_baseline_acc = y_val.mean()  # since winner=1 means higher seed won
print(f"\n=== Seed-Only Baseline ===")
print(f"Validation accuracy:  {seed_baseline_acc:.3f}")
print(f"\nBest model vs baseline improvement: {max(lr_val_acc, gb_val_acc) - seed_baseline_acc:+.3f}")

### 4.3 Model Evaluation

In [ ]:
# Pick the best model for evaluation
best_model = gb_model if gb_val_acc >= lr_val_acc else lr_model
best_name = "Gradient Boosting" if gb_val_acc >= lr_val_acc else "Logistic Regression"
print(f"Selected model: {best_name}\n")

# Predictions on validation set
y_pred = best_model.predict(X_val_scaled)
y_prob = best_model.predict_proba(X_val_scaled)[:, 1]

# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_val, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Lower Seed Won', 'Higher Seed Won'],
            yticklabels=['Lower Seed Won', 'Higher Seed Won'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title(f'{best_name} - Confusion Matrix')

# Predicted probability distribution
axes[1].hist(y_prob[y_val == 1], bins=15, alpha=0.7, label='Higher Seed Won', color='steelblue')
axes[1].hist(y_prob[y_val == 0], bins=15, alpha=0.7, label='Lower Seed Won (Upset)', color='coral')
axes[1].set_xlabel('Predicted Probability (Higher Seed Wins)')
axes[1].set_ylabel('Count')
axes[1].set_title('Predicted Probability Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

# Classification report
print(classification_report(y_val, y_pred, target_names=['Upset', 'Chalk']))

### 4.4 Feature Importance

In [ ]:
# Feature Importance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Logistic Regression coefficients (show direction of effect)
lr_importance = pd.Series(lr_model.coef_[0], index=feature_cols).sort_values()
lr_importance.plot(kind='barh', ax=axes[0], color=['coral' if v < 0 else 'steelblue' for v in lr_importance])
axes[0].set_title('Logistic Regression Coefficients')
axes[0].set_xlabel('Coefficient (positive = favors higher seed)')

# Gradient Boosting feature importance
gb_importance = pd.Series(gb_model.feature_importances_, index=feature_cols).sort_values()
gb_importance.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Gradient Boosting Feature Importance')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

print("Top 5 most important features (Gradient Boosting):")
for feat, imp in gb_importance.sort_values(ascending=False).head(5).items():
    print(f"  {feat:20s} {imp:.3f}")

---
## 5. Baseline Bracket (Seed-Based / Naive Model)

The naive baseline always picks the higher seed (lower number) to win. In cases of equal seeds, we default to the team listed first (arbitrary tiebreak).

**This bracket must be submitted alongside the model bracket.**

In [ ]:
# =============================================================
# Seed-Based Baseline Bracket Generator
# =============================================================
# Rule: The higher seed (lower number) ALWAYS wins.
# If seeds are equal, pick the first-listed team (arbitrary).

def generate_seed_bracket(bracket_teams):
    """
    Generate a bracket where the higher seed always wins.
    bracket_teams: list of dicts with 'team', 'seed', 'region' keys,
                   ordered by bracket position within each region.
    Returns round-by-round results.
    """
    results = {}
    
    # Group by region
    regions = {}
    for team in bracket_teams:
        r = team['region']
        if r not in regions:
            regions[r] = []
        regions[r].append(team)
    
    # Standard bracket order within a region (seed matchups)
    # Position 1v16, 8v9, 5v12, 4v13, 6v11, 3v14, 7v10, 2v15
    bracket_order = [1, 16, 8, 9, 5, 12, 4, 13, 6, 11, 3, 14, 7, 10, 2, 15]
    
    final_four = []
    
    for region_name, teams in regions.items():
        # Sort teams by bracket position
        seed_map = {t['seed']: t for t in teams}
        ordered = [seed_map[s] for s in bracket_order if s in seed_map]
        
        current_round = ordered
        round_num = 1
        results[f"{region_name}_R{round_num}"] = []
        
        while len(current_round) > 1:
            next_round = []
            for i in range(0, len(current_round), 2):
                if i + 1 < len(current_round):
                    a, b = current_round[i], current_round[i + 1]
                    # Higher seed (lower number) wins
                    winner = a if a['seed'] <= b['seed'] else b
                    results[f"{region_name}_R{round_num}"].append(
                        f"({winner['seed']}) {winner['team']} beats ({a['seed'] if winner != a else b['seed']}) {a['team'] if winner != a else b['team']}"
                    )
                    next_round.append(winner)
                else:
                    next_round.append(current_round[i])
            current_round = next_round
            round_num += 1
            if len(current_round) > 1:
                results[f"{region_name}_R{round_num}"] = []
        
        final_four.append(current_round[0])
        results[f"{region_name}_winner"] = f"({current_round[0]['seed']}) {current_round[0]['team']}"
    
    # Final Four
    results["Final Four"] = [f"({t['seed']}) {t['team']} [{t['region']}]" for t in final_four]
    
    # Semi-finals (1st region vs 2nd, 3rd vs 4th by bracket convention)
    if len(final_four) >= 4:
        sf1_winner = min(final_four[0], final_four[1], key=lambda t: t['seed'])
        sf2_winner = min(final_four[2], final_four[3], key=lambda t: t['seed'])
        results["Championship"] = f"({sf1_winner['seed']}) {sf1_winner['team']} vs ({sf2_winner['seed']}) {sf2_winner['team']}"
        champ = min(sf1_winner, sf2_winner, key=lambda t: t['seed'])
        results["Champion"] = f"({champ['seed']}) {champ['team']}"
    
    return results

print("Seed-based bracket generator ready.")
print("Will generate baseline bracket once 2026 matchups are announced on Selection Sunday (March 15).")
print("\\nFor now, the function is tested and ready to accept team data.")

---
## 6. Model Bracket Predictions (2026 Tournament)

Once the 2026 bracket is announced on Selection Sunday (March 15), we plug the matchups into our model to generate game-by-game predictions.

In [ ]:
# =============================================================
# Model-Based Bracket Generator
# =============================================================
# Uses the trained model to predict each game's winner based on
# stat differentials between the two teams.

def predict_matchup(team_a_stats, team_b_stats, model, scaler, features):
    """
    Predict the winner of a single matchup.
    Returns (winner_stats_dict, win_probability).
    """
    diff = {}
    for feat in features:
        val_a = team_a_stats.get(feat, 0)
        val_b = team_b_stats.get(feat, 0)
        if feat in ['Seed', 'AdjDE', 'TOPct']:
            diff[f'd_{feat}'] = val_b - val_a
        else:
            diff[f'd_{feat}'] = val_a - val_b
    
    X = pd.DataFrame([diff])
    X_scaled = scaler.transform(X)
    prob = model.predict_proba(X_scaled)[0][1]  # prob that team A wins
    
    if prob >= 0.5:
        return team_a_stats, prob
    else:
        return team_b_stats, 1 - prob


def generate_model_bracket(bracket_teams, team_stats_df, model, scaler, features):
    """
    Generate a full bracket using the trained model.
    bracket_teams: list of dicts with 'team', 'seed', 'region'
    team_stats_df: DataFrame with team stats (2025 season data)
    """
    results = {}
    bracket_order = [1, 16, 8, 9, 5, 12, 4, 13, 6, 11, 3, 14, 7, 10, 2, 15]
    
    # Map team names to their stats
    def get_team_stats(team_name):
        row = team_stats_df[team_stats_df['Mapped ESPN Team Name'] == team_name]
        if len(row) == 0:
            row = team_stats_df[team_stats_df['Full Team Name'].str.contains(team_name, case=False, na=False)]
        if len(row) == 0:
            return None
        return row.iloc[0][features].to_dict()
    
    regions = {}
    for team in bracket_teams:
        r = team['region']
        if r not in regions:
            regions[r] = []
        regions[r].append(team)
    
    final_four = []
    final_four_stats = []
    
    for region_name, teams in regions.items():
        seed_map = {t['seed']: t for t in teams}
        ordered = [seed_map[s] for s in bracket_order if s in seed_map]
        
        current_round = ordered
        current_stats = [get_team_stats(t['team']) or {f: 0 for f in features} for t in ordered]
        round_num = 1
        round_names = {1: 'Round of 64', 2: 'Round of 32', 3: 'Sweet 16', 4: 'Elite Eight'}
        
        results[f"{region_name} - {round_names.get(round_num, f'R{round_num}')}"] = []
        
        while len(current_round) > 1:
            next_round = []
            next_stats = []
            
            for i in range(0, len(current_round), 2):
                if i + 1 < len(current_round):
                    a, b = current_round[i], current_round[i + 1]
                    a_stats, b_stats = current_stats[i].copy(), current_stats[i + 1].copy()
                    
                    # Set seed in stats for prediction
                    a_stats['Seed'] = a['seed']
                    b_stats['Seed'] = b['seed']
                    
                    winner_stats, prob = predict_matchup(a_stats, b_stats, model, scaler, features)
                    
                    if winner_stats == a_stats:
                        winner, loser = a, b
                        next_stats.append(a_stats)
                    else:
                        winner, loser = b, a
                        next_stats.append(b_stats)
                    
                    results[f"{region_name} - {round_names.get(round_num, f'R{round_num}')}"].append(
                        f"({winner['seed']}) {winner['team']} over ({loser['seed']}) {loser['team']} [{prob:.1%}]"
                    )
                    next_round.append(winner)
                else:
                    next_round.append(current_round[i])
                    next_stats.append(current_stats[i].copy())
            
            current_round = next_round
            current_stats = next_stats
            round_num += 1
            if len(current_round) > 1:
                results[f"{region_name} - {round_names.get(round_num, f'R{round_num}')}"] = []
        
        final_four.append(current_round[0])
        final_four_stats.append(current_stats[0].copy())
        results[f"{region_name} Winner"] = f"({current_round[0]['seed']}) {current_round[0]['team']}"
    
    # Final Four
    results["Final Four"] = [f"({t['seed']}) {t['team']} [{t['region']}]" for t in final_four]
    
    if len(final_four) >= 4:
        # Semi-final 1
        ff1_stats, ff2_stats = final_four_stats[0].copy(), final_four_stats[1].copy()
        ff1_stats['Seed'] = final_four[0]['seed']
        ff2_stats['Seed'] = final_four[1]['seed']
        sf1_winner_stats, sf1_prob = predict_matchup(ff1_stats, ff2_stats, model, scaler, features)
        sf1_winner = final_four[0] if sf1_winner_stats == ff1_stats else final_four[1]
        sf1_loser = final_four[1] if sf1_winner == final_four[0] else final_four[0]
        
        # Semi-final 2
        ff3_stats, ff4_stats = final_four_stats[2].copy(), final_four_stats[3].copy()
        ff3_stats['Seed'] = final_four[2]['seed']
        ff4_stats['Seed'] = final_four[3]['seed']
        sf2_winner_stats, sf2_prob = predict_matchup(ff3_stats, ff4_stats, model, scaler, features)
        sf2_winner = final_four[2] if sf2_winner_stats == ff3_stats else final_four[3]
        sf2_loser = final_four[3] if sf2_winner == final_four[2] else final_four[2]
        
        results["Semi-Final 1"] = f"({sf1_winner['seed']}) {sf1_winner['team']} over ({sf1_loser['seed']}) {sf1_loser['team']} [{sf1_prob:.1%}]"
        results["Semi-Final 2"] = f"({sf2_winner['seed']}) {sf2_winner['team']} over ({sf2_loser['seed']}) {sf2_loser['team']} [{sf2_prob:.1%}]"
        
        # Championship
        champ_a_stats = sf1_winner_stats.copy()
        champ_b_stats = sf2_winner_stats.copy()
        champ_a_stats['Seed'] = sf1_winner['seed']
        champ_b_stats['Seed'] = sf2_winner['seed']
        champ_stats, champ_prob = predict_matchup(champ_a_stats, champ_b_stats, model, scaler, features)
        champion = sf1_winner if champ_stats == champ_a_stats else sf2_winner
        runner_up = sf2_winner if champion == sf1_winner else sf1_winner
        
        results["Championship"] = f"({champion['seed']}) {champion['team']} over ({runner_up['seed']}) {runner_up['team']} [{champ_prob:.1%}]"
        results["CHAMPION"] = f"({champion['seed']}) {champion['team']}"
    
    return results


def print_bracket(results):
    """Pretty print bracket results."""
    for round_name, value in results.items():
        print(f"\n{'='*60}")
        print(f"  {round_name}")
        print(f"{'='*60}")
        if isinstance(value, list):
            for game in value:
                print(f"  {game}")
        else:
            print(f"  {value}")


print("Model bracket generator ready.")
print("Will generate predictions once 2026 bracket is announced.")

# Prepare 2025 season stats for predictions
stats_2025 = df[df['Season'] == 2025].copy()
print(f"\n2025 season data available for {len(stats_2025)} teams")
print(f"Tournament teams in 2025 data: {len(stats_2025[stats_2025['Post-Season Tournament'] == 'March Madness'])}")

### Demo: Test Pipeline on 2025 Tournament Teams
Let's validate the full pipeline works by running predictions on the 2025 tournament teams that are already in our dataset.

In [ ]:
# Demo: Build bracket from 2025 tournament teams in our dataset
tourney_2025 = df[(df['Season'] == 2025) & (df['Post-Season Tournament'] == 'March Madness')].copy()

# Build team list from the data
demo_teams = []
for _, row in tourney_2025.iterrows():
    demo_teams.append({
        'team': row['Mapped ESPN Team Name'],
        'seed': int(row['Seed']),
        'region': row['Region']
    })

print(f"2025 Tournament Teams ({len(demo_teams)} teams):")
for region in tourney_2025['Region'].unique():
    region_teams = [t for t in demo_teams if t['region'] == region]
    region_teams.sort(key=lambda x: x['seed'])
    print(f"\n  {region}:")
    for t in region_teams:
        print(f"    ({t['seed']}) {t['team']}")

# --- Generate SEED-BASED bracket ---
print("\n" + "="*70)
print("  SEED-BASED BASELINE BRACKET (2025 Demo)")
print("="*70)
seed_results = generate_seed_bracket(demo_teams)
print_bracket(seed_results)

# --- Generate MODEL-BASED bracket ---
print("\n" + "="*70)
print("  MODEL-BASED BRACKET (2025 Demo)")
print("="*70)
model_results = generate_model_bracket(demo_teams, stats_2025, best_model, scaler, FEATURES)
print_bracket(model_results)

---
## 7. Post-Tournament Analysis & Model Revision

*This section will be completed after the tournament (due April 13, 2026)*

In [ ]:
# TODO (Post-Tournament):
# 1. Score both brackets against actual results
# 2. Analyze where the model was wrong — were there common patterns in mispredictions?
# 3. Revise the model (add/remove features, try different algorithms, adjust weights)
# 4. Justify changes with statistical reasoning

---
## 8. Reflection & Key Takeaways

*This section will be completed for the final report (due April 21, 2026)*

In [ ]:
# TODO (Final Report):
# - What worked well in the model?
# - What were the biggest challenges?
# - What would you do differently next time?
# - How did the model compare to the baseline?
# - Key statistical insights learned